Notebook: 02_signal_quality.ipynb

Purpose: Generate lead-level signal quality evidence using raw ECG waveforms only.

Inputs:
- raw ECG waveforms
- sampling rate

Outputs:
- signal_quality_features.parquet

Forbidden upstream evidence: delineation, QT measurement, confidence targets.

# 02 — Signal Quality Evidence

Compute canonical signal quality metrics at the record+lead level without using delineation or measurement evidence.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import signal
from scipy.fft import rfft, rfftfreq
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from ecg_analytics.datasets import PTBXLDataset, LUDBDataset, NSTDBDataset, INCARTDataset, QTDBDataset

root_dir = Path.cwd().parent
artifacts_dir = root_dir / 'artifacts'
artifacts_dir.mkdir(exist_ok=True)
data_dir = root_dir / 'data'

adapters = {
    'ptbxl': PTBXLDataset,
    'ludb': LUDBDataset,
    'nstdb': NSTDBDataset,
    'incart': INCARTDataset,
    'qtdb': QTDBDataset,
}

rows = []

for dataset_name, adapter_cls in adapters.items():
    dataset_path = data_dir / dataset_name
    if not dataset_path.exists():
        continue
    adapter = adapter_cls(data_dir=data_dir)
    try:
        record_names = adapter.list_records()
    except Exception:
        continue

    for record_name in record_names:
        try:
            record = adapter.load_record(record_name)
        except Exception:
            continue

        record_id = f'{dataset_name}/{record_name}'
        signal = np.asarray(record.signal, dtype=float)
        if signal.ndim == 1:
            signal = signal.reshape(-1, 1)
        fs = float(getattr(record, 'fs', np.nan))
        lead_names = getattr(record, 'lead_names', []) or []

        for lead_index, lead_name in enumerate(lead_names):
            lead_signal = signal[:, lead_index] if signal.shape[1] > lead_index else signal[:, 0]
            freq = rfftfreq(lead_signal.size, 1.0 / fs)
            spectrum = np.abs(rfft(lead_signal))
            band = (freq >= 40) & (freq <= 100)
            hfn_value = float(np.sum(spectrum[band]) / (np.sum(spectrum) + 1e-12)) if np.any(band) else 0.0
            powerline = 0.0
            for target in (50.0, 60.0):
                idx = int(np.argmin(np.abs(freq - target)))
                powerline += spectrum[idx] ** 2
            pli_value = float(powerline / (np.sum(spectrum ** 2) + 1e-12))
            clipped = np.abs(lead_signal - np.clip(lead_signal, np.percentile(lead_signal, 0.5), np.percentile(lead_signal, 99.5))) < 1e-6
            clipping_value = float(np.mean(clipped))
            flat_ratio = float(np.mean(np.abs(np.diff(lead_signal)) < max(1e-6, 0.01 * np.std(lead_signal)))) if lead_signal.size > 1 else 0.0
            rows.append({
                'record_id': record_id,
                'lead_id': str(lead_name).lower(),
                'bw_index': float(np.nan),
                'hfn_index': hfn_value,
                'pli_index': pli_value,
                'clipping_ratio': clipping_value,
                'flatline_ratio': flat_ratio,
                'signal_quality_score': float(np.clip(1.0 - hfn_value - pli_value - clipping_value, 0.0, 1.0)),
            })

signal_quality = pd.DataFrame(rows)
if not signal_quality.empty:
    signal_quality = signal_quality[['record_id','lead_id','bw_index','hfn_index','pli_index','clipping_ratio','flatline_ratio','signal_quality_score']]
    assert signal_quality[['record_id','lead_id']].duplicated().sum() == 0
    assert signal_quality['signal_quality_score'].between(0.0, 1.0).all()

signal_quality.to_parquet(artifacts_dir / 'signal_quality_features.parquet', index=False)
print('Wrote signal_quality_features.parquet')
